# GEPA walkthrough: two iterations, step by step

This notebook opens the GEPA loop on a tiny dataset so every step can be inspected.
Read [docs/gepa_guide.md](../docs/gepa_guide.md) alongside it.

**Data.** 5 synthetic certificates from the frozen GEPA buckets: 3 in `gepa_train` (GEPA's
*trainset*), 2 in `gepa_pareto` (GEPA's *valset*). Their gold values live in a separate
**demo store** (`notebooks/demo/annotations/`): pre-annotated by Haiku, then checked by
Claude against each image. They are **not** human gold and are never mixed into
`data/annotations/`.

**Cost.** A few cents (live OpenRouter calls); re-running is nearly free thanks to the disk cache.

**Plan.**
1. Setup and data
2. The candidate: what GEPA evolves
3. `evaluate`: run and score the seed prompt (with traces)
4. Which component to rewrite (`ErrorDrivenSelector`)
5. `make_reflective_dataset`: the feedback the reflection model reads
6. Reflection: GEPA's default proposer writes a new text
7. Acceptance: is the rewrite better on the same minibatch?
8. The real thing: `gepa.optimize` for 2 iterations, and how to read its logs

## 1. Setup and data

In [1]:
import json
import os
from pathlib import Path

if Path.cwd().name == "notebooks":  # run from the repo root
    os.chdir("..")

from vex.annotation.store import AnnotationStore, effective_split, held_out_ids
from vex.config import Settings, load_config
from vex.data.registry import load_registry
from vex.data.splits import load_splits
from vex.logs import setup_logging

setup_logging("WARNING")
NB_DIR = Path("runs/notebook")  # git-ignored scratch space for this notebook
config = load_config(Path("configs/default.yaml"), [
    "paths.annotations=notebooks/demo/annotations",   # the demo store, not the real one
    f"paths.prompts={NB_DIR / 'prompts'}",
    f"paths.review_queue={NB_DIR / 'review_queue'}",
    "gepa.max_metric_calls=18",                       # 2 + 2 iterations × (3 + 3 + 2): see section 8
    "gepa.reflection_minibatch_size=3",
    "gepa.use_merge=false",
    "budget.max_usd=1.0",
])
splits = load_splits(config.paths.splits)
store = AnnotationStore(config.paths.annotations)
registry = {r.doc_id: r for r in load_registry(config.paths.registry)}
gold = store.gold()

train_ids = effective_split("gepa_train", splits, store)    # bucket ∩ gold
val_ids = effective_split("gepa_pareto", splits, store)
assert not (set(train_ids) | set(val_ids)) & held_out_ids(splits)
for name, ids in (("trainset", train_ids), ("valset", val_ids)):
    print(name, [Path(registry[d].path).name for d in ids])

trainset ['syn_0003_attestation_lettre.jpg', 'syn_0010_memo.jpg', 'syn_0011_carte_internationale.jpg']
valset ['syn_0005_memo.jpg', 'syn_0014_attestation_tableau.jpg']


GEPA sees two datasets, and the difference matters:

- the **trainset** feeds *minibatches*: GEPA runs the current prompt on a few documents,
  shows the errors to a reflection model, and tests the rewrite on the same documents;
- the **valset** scores every *accepted* rewrite on all its documents. These scores choose
  which candidate to continue from (the Pareto front). Because they drive selection, the
  valset is **not** held out: our held-out sets (`heldout_blind`, `heldout_standin`) are
  scored only after GEPA has finished.

## 2. The candidate: what GEPA evolves

In [2]:
from vex.prompt import render, seed_candidate

seed = seed_candidate()
print(len(seed), "components:", list(seed))
print()
print("field:garanties =", seed["field:garanties"])

11 components: ['instructions', 'field:assureur', 'field:numero_contrat', 'field:nom_souscripteur', 'field:adresse_souscripteur', 'field:immatriculation', 'field:marque_vehicule', 'field:date_debut_validite', 'field:date_fin_validite', 'field:code_assureur', 'field:garanties']

field:garanties = Liste des garanties souscrites telles qu'énumérées sur le document (ex. « Responsabilité civile », « Vol », « Incendie », « Bris de glace », « Dommages tous accidents », « Assistance »), un élément par garantie, dans l'ordre du document, libellés recopiés sans montants ni franchises. Ne lister que les garanties marquées comme souscrites (cochées, « oui »). Si le document ne liste aucune garantie → null, jamais une liste vide.


A **candidate** is a `dict[str, str]`. Each key is a **component** GEPA can rewrite
independently: `instructions` (global rules) and one `field:<name>` per schema field.
`render` turns a candidate into what the extractor receives: a system prompt and a JSON
schema whose field descriptions come from the candidate.

In [3]:
rendered = render(seed)
print(rendered.system[:700], "...")

Tu extrais des champs d'un justificatif d'assurance automobile français : attestation d'assurance, Mémo Véhicule Assuré, carte internationale d'assurance automobile (blanche) ou ancienne carte verte. Les images sont les pages du document, éventuellement scannées, tournées ou de mauvaise qualité ; lis-les entièrement avant de répondre.
Pour chaque champ, suis sa description. Recopie les valeurs telles qu'elles apparaissent dans le document, sans rien inventer ni compléter. Si un champ n'apparaît pas ou est illisible, renvoie null, jamais une chaîne vide.
Réponds uniquement avec un objet JSON contenant exactement les champs demandés.

Champs à extraire :
- assureur : Nom de la compagnie d'assu ...


## 3. `evaluate`: run and score the seed prompt

`VexAdapter` is our implementation of GEPA's `GEPAAdapter` protocol
(`src/vex/gepa_adapter.py`). GEPA calls it synchronously, so the async OpenRouter client
lives on a background event loop.

In [4]:
from vex.diagnostic import Diagnostician
from vex.gepa_adapter import DocInstance, ErrorDrivenSelector, VexAdapter
from vex.llm.client import OpenRouterClient
from vex.llm.cost import CostTracker
from vex.llm.sync import BackgroundLoop

tracker = CostTracker(config.budget.max_usd)
loop = BackgroundLoop()
client = OpenRouterClient(config, Settings().openrouter_api_key, tracker)
adapter = VexAdapter(config, client, loop, Diagnostician(client, NB_DIR / "diagnostics", config.paths.review_queue))

def instances(ids):
    return [DocInstance(d, Path(registry[d].path), gold[d].values) for d in ids]

trainset, valset = instances(train_ids), instances(val_ids)
batch = adapter.evaluate(trainset, seed, capture_traces=True)
for traj in batch.trajectories:
    print(f"{Path(traj['path']).name:38} score={traj['score']:.3f}")
    for field in traj["fields"]:
        if field["error"] != "correct":
            print("    ", field["diff"])

syn_0003_attestation_lettre.jpg        score=1.000
syn_0010_memo.jpg                      score=0.857
     [omission (oubli)] nom_souscripteur: attendu/expected 'Virginie Launay-Lamy' — obtenu/got null
     [hallucination (valeur inventée)] garanties: attendu/expected null — obtenu/got ['ASSISTANCE 24H/24']
syn_0011_carte_internationale.jpg      score=1.000


Each document gets a score in [0, 1]: the weighted mean of its field scores
(`metric.weights`). GEPA only sees `batch.scores`. The **trajectories** are ours: GEPA passes
them back to `make_reflective_dataset` without looking inside. Each document evaluated costs
one **metric call** of the `max_metric_calls` budget.

## 4. Which component to rewrite

In [5]:
selector = ErrorDrivenSelector(config.metric.weights)
components = selector(None, batch.trajectories, batch.scores, 0, seed)
components

['field:nom_souscripteur']

GEPA rewrites one or a few components per iteration. Its default `round_robin` cycles
through all 11 components, even correct ones. Our `ErrorDrivenSelector` picks the field with
the largest weighted error on the minibatch (or `instructions` if an output could not be
parsed). Switch with `gepa.module_selector` in the YAML.

## 5. `make_reflective_dataset`: the feedback the reflection model reads

In [6]:
dataset = adapter.make_reflective_dataset(seed, batch, components)
print(json.dumps(dataset, ensure_ascii=False, indent=1)[:2500])

{
 "field:nom_souscripteur": [
  {
   "Inputs": {
    "field": "nom_souscripteur",
    "document": "syn_0010_memo.jpg"
   },
   "Generated Outputs": null,
   "Feedback": "[omission (oubli)] nom_souscripteur: attendu/expected 'Virginie Launay-Lamy' — obtenu/got null\nDans le document / In the document — En haut à droite du bandeau sombre, en page 1, sous le titre 'MÉMO VÉHICULE ASSURÉ' et à droite de 'Assur'Mirage' : « Virginie Launay-Lamy »\nCause : Le nom du souscripteur est présent en haut à droite de l'en-tête (bandeau sombre) sans étiquette explicite. L'extraction a renvoyé null, probablement parce que le nom n'est pas accompagné d'un libellé comme « Souscripteur » ou « Assuré », et que le modèle n'a pas associé ce texte isolé au titulaire du contrat."
  }
 ]
}


For every failing field of the chosen component, the `diagnostic` model (Haiku) looks at
the page and **must quote** where the expected value is. Unquoted answers are dropped. If it
cannot find the gold value at all, the case goes to `review_queue/suspect_annotations.jsonl`
(maybe the annotation is wrong) and is **not** taught to the prompt. Diagnoses are cached by
(doc, field, prediction, gold), so later iterations reuse them.

## 6. Reflection: GEPA's default proposer writes a new text

GEPA renders the reflective dataset into its default prompt template and calls our
`reflection_lm` (Haiku). Here we do by hand exactly what GEPA does internally.

In [7]:
from gepa.strategies.instruction_proposal import InstructionProposalSignature
from vex.optimize import ReflectionLM

reflection_lm = ReflectionLM(client, loop)
component = components[0]
prompt = InstructionProposalSignature.prompt_renderer({
    "current_instruction_doc": seed[component],
    "dataset_with_feedback": dataset[component],
    "prompt_template": None,
})
print(prompt[:1500], "...")

I provided an assistant with the following instructions to perform a task for me:
```
Nom du souscripteur, c'est-à-dire le titulaire du contrat (« Souscripteur », « Assuré », « Nom et adresse de l'assuré », case 8 de la carte internationale) : prénom et nom d'une personne (sans civilité M., Mme) ou raison sociale d'une entreprise. Si un conducteur désigné est aussi mentionné, prendre le souscripteur. Si le champ est absent ou illisible → null (jamais de chaîne vide).
```

The following are examples of different task inputs provided to the assistant along with the assistant's response for each of them, and some feedback on how the assistant's response could be better:
```
# Example 1
## Inputs
### field
nom_souscripteur

### document
syn_0010_memo.jpg

## Generated Outputs
None

## Feedback
[omission (oubli)] nom_souscripteur: attendu/expected 'Virginie Launay-Lamy' — obtenu/got null
Dans le document / In the document — En haut à droite du bandeau sombre, en page 1, sous le titre 'MÉMO 

In [8]:
reply = reflection_lm(prompt)
new_text = InstructionProposalSignature.output_extractor(reply)["new_instruction"]
print("BEFORE:", seed[component])
print()
print("AFTER: ", new_text)

BEFORE: Nom du souscripteur, c'est-à-dire le titulaire du contrat (« Souscripteur », « Assuré », « Nom et adresse de l'assuré », case 8 de la carte internationale) : prénom et nom d'une personne (sans civilité M., Mme) ou raison sociale d'une entreprise. Si un conducteur désigné est aussi mentionné, prendre le souscripteur. Si le champ est absent ou illisible → null (jamais de chaîne vide).

AFTER:  Ta tâche consiste à extraire, à partir d'un document d'assurance automobile (carte verte / carte internationale d'assurance automobile, mémo véhicule assuré, attestation ou document similaire), la valeur du champ **nom_souscripteur**.

**Définition du champ**
- Le champ désigne le souscripteur, c'est-à-dire le titulaire du contrat d'assurance. Il correspond aux intitulés suivants : « Souscripteur », « Assuré », « Nom et adresse de l'assuré » (case 8 de la carte internationale d'assurance automobile).
- La valeur est soit le prénom et le nom d'une personne physique, soit la raison sociale d'

## 7. Acceptance: is the rewrite better on the same minibatch?

In [9]:
child = {**seed, component: new_text}
child_batch = adapter.evaluate(trainset, child, capture_traces=False)
print("parent minibatch sum:", round(sum(batch.scores), 3))
print("child  minibatch sum:", round(sum(child_batch.scores), 3))
print("accepted" if sum(child_batch.scores) > sum(batch.scores) else "rejected (GEPA requires a strict improvement)")

parent minibatch sum: 2.857
child  minibatch sum: 3.0
accepted


GEPA compares **sums** on the same minibatch (`acceptance_criterion="strict_improvement"`).
Only an accepted child is then scored on the full valset (more metric calls) and added to the
candidate pool. That is one iteration: 3 + 3 metric calls here, plus 2 if accepted.

Note what the selector did in section 4: the Mémo had two errors of equal weight (an omitted
name, an invented "assistance" guarantee); ties go to schema order, so `nom_souscripteur` was
rewritten. A longer, more explicit prompt can also fix other fields as a side effect: check
the child's per-field results, not only its sum.

## 8. The real thing: `gepa.optimize` for 2 iterations

`run_optimize` wires everything into `gepa.optimize` (`src/vex/optimize.py`): trainset,
valset, adapter, `reflection_lm`, our selector, the budget stopper and a callback that writes
the run logs. Budget: 2 metric calls for the seed's valset score, then up to 3 + 3 + 2 per
iteration, so `max_metric_calls=18` allows two full iterations. GEPA checks the budget
*between* iterations, so the last one can overshoot a little.

In [10]:
from vex.optimize import run_optimize
from vex.runs import create_run_dir

loop.run(client.aclose()); loop.close()  # run_optimize builds its own client and loop
run_dir = create_run_dir(config, "notebook-optimize")
setup_logging("WARNING")
outcome = run_optimize(config, Settings(), seed, run_dir)
print("run dir:", run_dir)
print("cost so far in run_optimize: $%.4f" % outcome.total_usd)

21:17:08 INFO    Run directory: runs/20261017T191708741658Z-notebook-optimize-aef58709 (config aef58709)


21:17:09 WARNING heldout_blind: no gold document yet (annotate it blind first); not scored


21:17:09 WARNING heldout_standin: no gold document yet (annotate it blind first); not scored


Iteration 0: Base program full valset score: 1.0 over 2 / 2 examples
Iteration 1: Selected program 0 score: 1.0
Iteration 1: Proposed new text for field:nom_souscripteur: Ta tâche consiste à extraire, à partir d'un document d'assurance automobile (carte verte / carte internationale d'assurance automobile, mémo véhicule assuré, attestation ou document similaire), la valeur du champ **nom_souscripteur**.

**Définition du champ**
- Le champ désigne le souscripteur, c'est-à-dire le titulaire du contrat d'assurance. Il correspond aux intitulés suivants : « Souscripteur », « Assuré », « Nom et adresse de l'assuré » (case 8 de la carte internationale d'assurance automobile).
- La valeur est soit le prénom et le nom d'une personne physique, soit la raison sociale d'une entreprise.
- Ne pas inclure de civilité (M., Mme, Mlle, etc.).
- Si un conducteur désigné est aussi mentionné, retenir le souscripteur et non le conducteur.

**Règles de recherche**
- Examine l'intégralité du document, y compri

### Reading the run logs

- `iterations.jsonl`: one line per event: `accepted` / `rejected` (minibatch scores) and
  `end` (total metric calls used so far);
- `proposals.jsonl`: the new texts the reflection model proposed;
- `candidates.jsonl`: every candidate scored on the full valset, with its average;
- `gepa/`: GEPA's own state (`gepa_state.bin`, resumable with `vex optimize --resume`).


In [11]:
for line in (run_dir / "iterations.jsonl").read_text().splitlines():
    event = json.loads(line)
    event.pop("usd_so_far")
    print(event)

{'iteration': 1, 'event': 'accepted', 'new_candidate_idx': 1, 'minibatch_score': 3.0}
{'iteration': 1, 'event': 'end', 'proposal_accepted': True, 'total_metric_calls': 8}
{'iteration': 2, 'event': 'end', 'proposal_accepted': False, 'total_metric_calls': 11}
{'iteration': 3, 'event': 'end', 'proposal_accepted': False, 'total_metric_calls': 14}
{'iteration': 4, 'event': 'end', 'proposal_accepted': False, 'total_metric_calls': 17}
{'iteration': 5, 'event': 'end', 'proposal_accepted': False, 'total_metric_calls': 20}


In [12]:
for line in (run_dir / "candidates.jsonl").read_text().splitlines():
    entry = json.loads(line)
    print(f"candidate {entry['candidate_idx']} (iteration {entry['iteration']}): "
          f"valset average {entry['average_score']:.3f}, parents {entry['parent_ids']}")

candidate 0 (iteration 0): valset average 1.000, parents []
candidate 1 (iteration 1): valset average 1.000, parents [0]


In [13]:
import difflib

for key in seed:
    if outcome.best[key] != seed[key]:
        print(f"--- {key}")
        print("\n".join(difflib.unified_diff(seed[key].split(". "), outcome.best[key].split(". "), lineterm="")))
print("best == seed:", outcome.best == seed)
print("held-out:", {split: data["n"] for split, data in outcome.heldout.items()}, "(no blind gold in the demo store)")

best == seed: True
held-out: {'heldout_blind': 0, 'heldout_standin': 0} (no blind gold in the demo store)


In [14]:
# The accepted candidate (index 1), compared with the seed:
accepted = [json.loads(line) for line in (run_dir / "candidates.jsonl").read_text().splitlines()][-1]["candidate"]
for key in seed:
    if accepted[key] != seed[key]:
        print(f"--- {key} (seed)\n{seed[key]}\n\n+++ {key} (candidate 1)\n{accepted[key][:1200]}")

--- field:nom_souscripteur (seed)
Nom du souscripteur, c'est-à-dire le titulaire du contrat (« Souscripteur », « Assuré », « Nom et adresse de l'assuré », case 8 de la carte internationale) : prénom et nom d'une personne (sans civilité M., Mme) ou raison sociale d'une entreprise. Si un conducteur désigné est aussi mentionné, prendre le souscripteur. Si le champ est absent ou illisible → null (jamais de chaîne vide).

+++ field:nom_souscripteur (candidate 1)
Ta tâche consiste à extraire, à partir d'un document d'assurance automobile (carte verte / carte internationale d'assurance automobile, mémo véhicule assuré, attestation ou document similaire), la valeur du champ **nom_souscripteur**.

**Définition du champ**
- Le champ désigne le souscripteur, c'est-à-dire le titulaire du contrat d'assurance. Il correspond aux intitulés suivants : « Souscripteur », « Assuré », « Nom et adresse de l'assuré » (case 8 de la carte internationale d'assurance automobile).
- La valeur est soit le prénom e

### What to take away

- **Iteration 1** found and fixed a real error: the rewrite was accepted on the minibatch.
- **But the valset could not tell the two candidates apart**: the seed already scored 1.0 on
  its 2 documents, so did the child, and GEPA keeps the first candidate on ties, so "best" is
  still the seed. The Pareto valset needs enough (and hard enough) documents for selection to
  mean anything; with real rounds, `gepa_pareto` grows with every reviewed batch.
- **Iterations 2-5 were skipped**: once a candidate is perfect on the minibatch, GEPA skips
  reflection (`skip_perfect_score`), yet each skip still costs 3 metric calls. A perfect
  trainset means "annotate more, harder documents", not "run more iterations".
- Improvements are first visible on minibatches (`iterations.jsonl`), then confirmed or not
  on the valset (`candidates.jsonl`), and finally checked on held-out by `vex optimize`.
- Stop when the held-out score plateaus across rounds (P6: `vex report`), not when the valset
  score stops moving: the valset is optimized on, so it is optimistic.